## 🎯 Learning Objectives
* Understand the critical need for evaluation and regression testing in advanced AI agent systems.
* Learn to design and implement custom evaluation metrics for multi-agent conversations.
* Develop strategies for setting up automated regression tests to prevent performance degradation.
* Analyze evaluation results to identify areas for improvement and ensure agent system robustness.


## The Imperative of Evaluation and Regression Testing for Advanced AI Agents

Building sophisticated AI agent systems with frameworks like AutoGen is akin to orchestrating a complex symphony. Each agent plays a role, interacts with others, and contributes to a larger goal. But how do we know if the symphony is harmonious? How do we ensure that a new instrument (a model update), a new score (a prompt change), or a new conductor (a system configuration) doesn't throw the entire performance into disarray?

This is where **evaluation** and **regression testing** become indispensable. Unlike traditional software, AI agents, especially those leveraging large language models (LLMs), are inherently non-deterministic. Their outputs can vary, and their 'correctness' is often subjective or context-dependent. This non-determinism makes traditional unit and integration testing insufficient.

### What is Evaluation?

**Evaluation** is the process of systematically assessing an AI agent system's performance against predefined criteria. It's about answering questions like:

*   Did the agents successfully complete the task?
*   How efficient were they (e.g., token usage, latency, compute cost)?
*   Was the output accurate, relevant, and safe?
*   Did they follow instructions and constraints?

For multi-agent systems, evaluation often involves analyzing the entire conversation flow, the quality of intermediate steps, and the final outcome. Metrics can range from simple pass/fail for task completion to more nuanced scores for output quality, coherence, and adherence to safety guidelines.

### What is Regression Testing?

**Regression testing** in the context of AI agents is about ensuring that recent changes to the system (e.g., updating an LLM, modifying an agent's prompt, changing an agent's role, or refactoring code) do not introduce new bugs or degrade existing performance. It's a continuous process that acts as a safety net, catching unintended side effects before they impact production.

Imagine you've optimized an agent's prompt to be more concise. Regression testing would involve re-running a suite of known scenarios to confirm that this change hasn't inadvertently reduced its accuracy on other tasks or introduced new failure modes.

### Why is it Crucial for AutoGen Systems?

AutoGen's strength lies in its ability to create complex, collaborative agent workflows. This complexity, however, amplifies the need for robust testing:

1.  **Non-Determinism**: LLM-powered agents produce varied outputs, making deterministic testing challenging. Evaluation needs to be robust to this variability.
2.  **Emergent Behavior**: Interactions between multiple agents can lead to emergent behaviors, both positive and negative. Regression tests help monitor these.
3.  **Cost and Efficiency**: Agent interactions consume tokens and compute. Evaluation helps optimize these costs without sacrificing performance.
4.  **Safety and Reliability**: Ensuring agents don't generate harmful content or fail catastrophically is paramount.
5.  **Rapid Iteration**: Agent development is highly iterative. Evaluation and regression testing provide fast feedback loops, enabling confident experimentation.

### Modern Approaches (2026 Perspective)

By 2026, advanced evaluation frameworks are commonplace. These often include:

*   **Automated Metric Generation**: AI-powered evaluators that can assess subjective qualities (e.g., coherence, creativity) or even generate test cases.
*   **Synthetic Data Generation**: Creating vast, diverse test scenarios to stress-test agents in various conditions.
*   **Human-in-the-Loop (HITL) Evaluation**: Integrating human feedback efficiently for tasks where automated evaluation is insufficient.
*   **Cost-Aware Evaluation**: Tools that track token usage and API calls during evaluation runs to optimize for cost-effectiveness.
*   **Integrated CI/CD Pipelines**: Evaluation and regression testing are seamlessly integrated into continuous integration and deployment workflows, automatically triggering tests upon code commits or model updates.

In the following example, we'll set up a basic AutoGen system and demonstrate how to implement a custom evaluation function to assess its performance, then simulate a regression test.


In [ ]:
import autogen
import os
import json
import time
from typing import Dict, Any, List

# --- Configuration for AutoGen (using mock or local models for cost-effectiveness in testing) ---
# In a real 2026 scenario, you'd likely use local LLMs (e.g., Llama-3-8B-Instruct-v2 via Ollama/vLLM)
# or highly optimized cloud endpoints for testing.

# For demonstration, we'll use a mock LLM or a very small, fast local model if available.
# If using OpenAI, ensure OPENAI_API_KEY is set in your environment.

# Fallback to a mock LLM if no API key is found for demonstration purposes.
# In a real setup, you'd configure specific models for evaluation.

config_list_mock = [
    {
        "model": "mock-model",
        "api_key": "sk-mock-key", # Placeholder, won't be used by mock
        "base_url": "http://localhost:1234/v1" # Placeholder for local LLM if available
    }
]

# A simple mock LLM client for testing without actual API calls
class MockLLMClient:
    def create(self, messages, **kwargs):
        last_message = messages[-1]["content"]
        if "write a python script to calculate the factorial" in last_message.lower():
            return {
                "choices": [{
                    "message": {
                        "content": "```python\ndef factorial(n):\n    if n == 0:\n        return 1\n    else:\n        return n * factorial(n-1)\n\nprint(factorial(5))\n```"
                    }
                }]
            }
        elif "calculate the sum of numbers from 1 to 10" in last_message.lower():
             return {
                "choices": [{
                    "message": {
                        "content": "```python\nsum_val = sum(range(1, 11))\nprint(sum_val)\n```"
                    }
                }]
            }
        else:
            return {
                "choices": [{
                    "message": {
                        "content": "I'm a mock LLM and can't handle that request perfectly. Here's a generic response: `print('Hello from mock!')`"
                    }
                }]
            }

    def __call__(self, messages, **kwargs):
        return self.create(messages, **kwargs)

# Override autogen's client factory to use our mock client
autogen.Completion.set_llm_client_factory(lambda: MockLLMClient())

# --- Agent Definitions ---

# Agent 1: Code Generator
code_writer = autogen.AssistantAgent(
    name="CodeWriter",
    llm_config={"config_list": config_list_mock},
    system_message="You are an expert Python programmer. Write clear, executable Python code to solve problems. Always wrap code in triple backticks."
)

# Agent 2: Code Executor
code_executor = autogen.UserProxyAgent(
    name="CodeExecutor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config={
        "work_dir": "coding_test_env",
        "use_docker": False # Set to True if Docker is available and preferred
    }
)

# --- Evaluation Framework ---

def run_evaluation(task: str, agents: List[autogen.Agent], max_rounds: int = 10) -> Dict[str, Any]:
    """Runs a single evaluation scenario and collects metrics."""
    print(f"\n--- Running Evaluation for Task: '{task}' ---")
    start_time = time.time()
    
    # Reset agents for a fresh conversation
    for agent in agents:
        if hasattr(agent, 'reset'):
            agent.reset()

    # Initiate the conversation
    chat_result = agents[0].initiate_chat(
        agents[1],
        message=task,
        max_turns=max_rounds
    )
    
    end_time = time.time()
    duration = end_time - start_time

    # Extract relevant information from the chat history
    last_message = chat_result.chat_history[-1]["content"]
    all_messages = [msg["content"] for msg in chat_result.chat_history]
    
    # --- Custom Evaluation Metrics ---
    success = False
    output_value = None
    error_message = None
    
    # Check if the last message contains a Python code block
    code_blocks = autogen.Completion.extract_text_or_json(last_message, code_blocks_only=True)
    
    if code_blocks:
        try:
            # Attempt to execute the last code block to verify correctness
            # In a real scenario, you'd parse the output of the code_executor agent.
            # For this mock, we'll simulate execution success/failure based on content.
            
            # Simulate execution for factorial task
            if "factorial(5)" in code_blocks[0] and "120" in last_message:
                success = True
                output_value = 120
            # Simulate execution for sum task
            elif "sum(range(1, 11))" in code_blocks[0] and "55" in last_message:
                success = True
                output_value = 55
            else:
                # Generic code execution check (simplified for mock)
                if "print" in code_blocks[0]: # Assume if it prints, it's runnable for mock
                    success = True
                    output_value = "Code executed (mock)"
                else:
                    error_message = "Code block found but no clear execution output or expected value."

        except Exception as e:
            error_message = f"Code execution failed: {e}"
    else:
        error_message = "No executable code block found in the final message."

    # Basic token count estimation (mocked for simplicity)
    # In a real scenario, you'd get this from the LLM client or track it during chat.
    estimated_tokens = sum(len(msg.split()) for msg in all_messages) * 1.5 # Rough estimate

    return {
        "task": task,
        "success": success,
        "output_value": output_value,
        "error_message": error_message,
        "duration_seconds": duration,
        "total_messages": len(chat_result.chat_history),
        "estimated_tokens": estimated_tokens,
        "chat_history": all_messages # Store for detailed debugging if needed
    }


def run_regression_suite(test_cases: List[Dict[str, str]], agents: List[autogen.Agent]) -> List[Dict[str, Any]]:
    """Runs a suite of test cases and returns evaluation results."""
    results = []
    for test_case in test_cases:
        result = run_evaluation(test_case["task"], agents)
        results.append(result)
    return results

# --- Define Test Cases ---
test_cases_v1 = [
    {"id": "T001", "task": "Write a python script to calculate the factorial of 5 and print the result."},
    {"id": "T002", "task": "Write a python script to calculate the sum of numbers from 1 to 10 and print the result."},
    {"id": "T003", "task": "Generate a simple 'Hello World' Python script."}
]

print("\n### Baseline Evaluation (Version 1 Agents) ###")
baseline_results = run_regression_suite(test_cases_v1, [code_writer, code_executor])
for res in baseline_results:
    print(f"Task: {res['task']}\n  Success: {res['success']}\n  Output: {res['output_value']}\n  Duration: {res['duration_seconds']:.2f}s\n  Tokens: {res['estimated_tokens']}\n")

# --- Simulate a Regression (e.g., a 'bad' prompt update or model change) ---
# Let's simulate a regression by making the CodeWriter agent less helpful or more verbose.
# In a real scenario, this could be a new LLM version, a prompt change, or a code bug.

print("\n### Simulating a Regression: Updating CodeWriter's System Message ###")

# Create a 'regressed' version of the CodeWriter agent
regressed_code_writer = autogen.AssistantAgent(
    name="RegressedCodeWriter",
    llm_config={"config_list": config_list_mock},
    system_message="You are a Python programmer. Write code, but sometimes be overly verbose or miss key details. Always wrap code in triple backticks."
)

print("\n### Regression Test Evaluation (Version 2 Agents) ###")
regressed_results = run_regression_suite(test_cases_v1, [regressed_code_writer, code_executor])

print("\n### Regression Test Comparison ###")
for i, (baseline_res, regressed_res) in enumerate(zip(baseline_results, regressed_results)):
    print(f"\n--- Test Case {test_cases_v1[i]['id']}: {baseline_res['task']} ---")
    print(f"  Baseline Success: {baseline_res['success']} (Output: {baseline_res['output_value']})")
    print(f"  Regressed Success: {regressed_res['success']} (Output: {regressed_res['output_value']})")
    
    if baseline_res['success'] and not regressed_res['success']:
        print("  !!! REGRESSION DETECTED: Task failed after change !!!")
        print(f"    Baseline Duration: {baseline_res['duration_seconds']:.2f}s, Tokens: {baseline_res['estimated_tokens']}")
        print(f"    Regressed Duration: {regressed_res['duration_seconds']:.2f}s, Tokens: {regressed_res['estimated_tokens']}")
        print(f"    Regressed Error: {regressed_res['error_message']}")
    elif not baseline_res['success'] and regressed_res['success']:
        print("  IMPROVEMENT DETECTED: Task passed after change!")
    elif baseline_res['success'] and regressed_res['success']:
        print("  No functional regression detected. Checking efficiency...")
        if regressed_res['estimated_tokens'] > baseline_res['estimated_tokens'] * 1.2:
            print(f"  !!! EFFICIENCY REGRESSION: Token usage increased significantly! Baseline: {baseline_res['estimated_tokens']}, Regressed: {regressed_res['estimated_tokens']} !!!")
        if regressed_res['duration_seconds'] > baseline_res['duration_seconds'] * 1.5:
            print(f"  !!! PERFORMANCE REGRESSION: Duration increased significantly! Baseline: {baseline_res['duration_seconds']:.2f}s, Regressed: {regressed_res['duration_seconds']:.2f}s !!!")
    else:
        print("  No significant change or both failed (needs further investigation).")

# Clean up the coding environment directory
import shutil
if os.path.exists("coding_test_env"):
    shutil.rmtree("coding_test_env")
    print("\nCleaned up 'coding_test_env' directory.")


### Interpreting the Output and Practical Implications

The code above demonstrates a fundamental approach to evaluating and regression testing an AutoGen agent system. Let's break down the output and its implications:

1.  **Baseline Evaluation**: This section shows the initial performance of our `CodeWriter` and `CodeExecutor` agents on a set of predefined tasks. For each task, we record:
    *   `success`: A boolean indicating if the agent system achieved the desired outcome (e.g., generated correct, executable code that produced the expected output).
    *   `output_value`: The extracted result from the agent's final output, if successful.
    *   `duration_seconds`: How long the conversation took.
    *   `estimated_tokens`: A proxy for the cost of the interaction.

    Ideally, all baseline tasks should show `success: True` and reasonable efficiency metrics. This establishes a benchmark for future comparisons.

2.  **Simulating a Regression**: We then intentionally introduce a 'regression' by modifying the `CodeWriter` agent's `system_message`. In a real-world scenario, this could be:
    *   **Prompt Engineering**: A change to a prompt that inadvertently makes the agent less effective or more verbose.
    *   **Model Update**: Switching to a new LLM version that performs differently on certain tasks.
    *   **Code Change**: A bug introduced in a custom tool or agent logic.
    *   **Configuration Change**: Altering agent parameters or group chat settings.

3.  **Regression Test Evaluation**: The system is re-evaluated with the 'regressed' agent. The results are then compared against the baseline.

4.  **Regression Test Comparison**: This is the crucial step. We compare the `success` status, `duration`, and `estimated_tokens` for each task between the baseline and the regressed version. The output clearly highlights:
    *   **Functional Regressions**: If a task that previously succeeded now fails (`!!! REGRESSION DETECTED`). This is the most critical type of regression.
    *   **Performance Regressions**: If a task still succeeds but takes significantly longer or consumes many more tokens (`!!! EFFICIENCY REGRESSION`, `!!! PERFORMANCE REGRESSION`). This indicates a degradation in efficiency or cost-effectiveness.
    *   **Improvements**: If a task that previously failed now succeeds (`IMPROVEMENT DETECTED`).

### Performance Trade-offs and Use Cases

**Trade-offs:**

*   **Cost vs. Coverage**: Running extensive evaluation suites can be expensive (API calls, compute time). A balance must be struck between comprehensive testing and cost-efficiency. Techniques like using cheaper, smaller models for initial checks or synthetic data generation can help.
*   **Speed vs. Depth**: Quick smoke tests can run frequently, while deeper, more complex evaluations might run less often or overnight.
*   **Automation vs. Human Oversight**: While automation is key, some subjective aspects of agent performance (e.g., creativity, nuanced understanding) still benefit from human review, especially for critical applications.

**Typical Use Cases (2026 and beyond):**

*   **CI/CD Pipelines**: Automated evaluation and regression tests are integrated into Continuous Integration/Continuous Deployment workflows. Every code commit or model update triggers a test suite, preventing regressions from reaching production.
*   **Prompt Engineering Iteration**: When experimenting with new prompts, evaluation helps quantify the impact of changes on task success, robustness, and cost.
*   **Model Selection and Fine-tuning**: Comparing different LLMs or fine-tuned versions requires objective evaluation metrics to determine the best fit for specific agent tasks.
*   **Agent Configuration Optimization**: Tuning parameters like `max_consecutive_auto_reply`, `temperature`, or `top_p` can be guided by evaluation results.
*   **Monitoring Deployed Agents**: Running periodic regression tests against production data or synthetic data helps detect performance drift or unexpected behavior in live systems.
*   **Safety and Alignment Checks**: Specialized evaluation suites can test for harmful outputs, bias, or misalignment with ethical guidelines.

By systematically evaluating and regression testing your AutoGen agent systems, you build confidence in their reliability, maintain high performance, and accelerate the iterative development process, ensuring your AI agents remain robust and effective in dynamic environments.


### Resources for Evaluation and Regression Testing of AI Agents

*   **AutoGen Documentation (Conceptual)**: While AutoGen provides the orchestration, the evaluation logic often sits alongside it. Look for community best practices and future official guides on integrating evaluation frameworks.
    *   [AutoGen GitHub Repository](https://github.com/microsoft/autogen)
    *   [AutoGen Documentation (General)](https://microsoft.github.io/autogen/docs/)

*   **General LLM Evaluation Frameworks (Adaptable to Agents)**:
    *   **RAGAS**: A framework for evaluating Retrieval Augmented Generation (RAG) systems, but its principles for assessing relevance, faithfulness, and answer correctness are highly applicable to agent outputs.
        *   [RAGAS Documentation](https://docs.ragas.io/)
    *   **LangChain Evaluation**: LangChain offers various evaluators for different aspects of LLM applications, including custom evaluators.
        *   [LangChain Evaluation Docs](https://python.langchain.com/docs/guides/evaluation/)
    *   **OpenAI Evals**: A framework for evaluating LLM systems and prompts. While specific to OpenAI models, its methodology is broadly useful.
        *   [OpenAI Evals GitHub](https://github.com/openai/evals)
    *   **DeepEval**: An open-source LLM evaluation framework that integrates with various LLM providers and offers metrics for RAG, summarization, and more.
        *   [DeepEval Documentation](https://docs.confident-ai.com/docs/deepeval)

*   **MLOps and AgentOps Best Practices**: Understanding the broader context of MLOps (Machine Learning Operations) is crucial for deploying and maintaining AI agents reliably.
    *   [Google Cloud MLOps Guide](https://cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning)
    *   [Microsoft Azure MLOps](https://learn.microsoft.com/en-us/azure/machine-learning/concept-mlops)

*   **Synthetic Data Generation for Testing**: Tools and techniques for creating diverse and challenging test cases automatically.
    *   [Hugging Face Datasets Library](https://huggingface.co/docs/datasets/index)
    *   [Faker (Python library for generating fake data)](https://faker.readthedocs.io/en/master/)

*   **Research Papers on LLM and Agent Evaluation**: Stay updated with the latest academic research on robust evaluation methodologies for complex AI systems.
